In [ ]:
# Setup: minimal in-process Django settings (no project or settings module needed).
import django
from django.conf import settings

if not settings.configured:
    settings.configure(
        DEBUG=True,
        DATABASES={'default': {'ENGINE': 'django.db.backends.sqlite3', 'NAME': ':memory:'}},
        INSTALLED_APPS=['django.contrib.contenttypes', 'django.contrib.auth'],
        ROOT_URLCONF=__name__,
        SECRET_KEY='test-secret-key-not-for-production',
        USE_TZ=True,
    )
django.setup()
print(f'Django {django.get_version()} configured')

In [ ]:
# Example 1: URL routing with path() + reverse()
from django.urls import path, reverse, resolve
from django.http import HttpResponse

def hello_view(request):
    return HttpResponse('hello')

def item_view(request, item_id):
    return HttpResponse(f'item {item_id}')

urlpatterns = [
    path('hello/', hello_view, name='hello'),
    path('items/<int:item_id>/', item_view, name='item'),
]

# Patch the URL conf for resolution
import sys
sys.modules[__name__].urlpatterns = urlpatterns

hello_url = reverse('hello')
item_url = reverse('item', kwargs={'item_id': 42})
print(f'hello_url = {hello_url}')
print(f'item_url = {item_url}')

match = resolve('/items/42/')
print(f'resolved view: {match.func.__name__}, kwargs={match.kwargs}')
assert hello_url == '/hello/'
assert item_url == '/items/42/'
assert match.kwargs == {'item_id': 42}


In [ ]:
# Example 2: HttpRequest / HttpResponse + RequestFactory for testing
from django.test import RequestFactory
from django.http import HttpResponse, JsonResponse

factory = RequestFactory()

def echo_view(request):
    return JsonResponse({
        'method': request.method,
        'path': request.path,
        'query': dict(request.GET.items()),
    })

request = factory.get('/echo/?foo=bar&baz=1')
response = echo_view(request)
print(f'status_code = {response.status_code}')
print(f'content = {response.content.decode()}')

import json
payload = json.loads(response.content)
assert payload['method'] == 'GET'
assert payload['path'] == '/echo/'
assert payload['query'] == {'foo': 'bar', 'baz': '1'}


In [ ]:
# Example 3: Template rendering via Django Template engine
from django.template import Template, Context, Engine

engine = Engine()
tmpl = engine.from_string(
    'Hello, {{ name|upper }}! You have {{ count }} message{{ count|pluralize }}.'
)
rendered = tmpl.render(Context({'name': 'world', 'count': 3}))
print(rendered)
assert rendered == 'Hello, WORLD! You have 3 messages.'

rendered_one = tmpl.render(Context({'name': 'alice', 'count': 1}))
print(rendered_one)
assert rendered_one == 'Hello, ALICE! You have 1 message.'


In [ ]:
# Example 4: Form validation
from django import forms

class ContactForm(forms.Form):
    name = forms.CharField(max_length=50)
    email = forms.EmailField()
    age = forms.IntegerField(min_value=0, max_value=150)

# Valid form
valid_form = ContactForm(data={'name': 'Alice', 'email': 'a@example.com', 'age': 30})
print(f'valid_form.is_valid() = {valid_form.is_valid()}')
print(f'cleaned_data = {valid_form.cleaned_data}')
assert valid_form.is_valid()

# Invalid form
bad_form = ContactForm(data={'name': 'Bob', 'email': 'not-an-email', 'age': 999})
print(f'bad_form.is_valid() = {bad_form.is_valid()}')
print(f'errors = {dict(bad_form.errors)}')
assert not bad_form.is_valid()
assert 'email' in bad_form.errors
assert 'age' in bad_form.errors


In [ ]:
# Example 5: ORM with in-memory SQLite — create model, migrate, save/query
# Jupyter's kernel runs inside an asyncio event loop, so Django's async-unsafe guard would
# raise SynchronousOnlyOperation on any DB cursor. DJANGO_ALLOW_ASYNC_UNSAFE is Django's
# documented escape hatch for notebook environments.
import os

os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

from django.db import models, connection

class Widget(models.Model):
    name = models.CharField(max_length=50)
    weight = models.IntegerField(default=0)

    class Meta:
        app_label = 'contenttypes'  # piggyback on an already-installed app

with connection.schema_editor() as editor:
    editor.create_model(Widget)

Widget.objects.create(name='alpha', weight=10)
Widget.objects.create(name='beta', weight=20)
Widget.objects.create(name='gamma', weight=30)

total = Widget.objects.count()
heavy = list(Widget.objects.filter(weight__gte=20).order_by('name').values_list('name', flat=True))
agg = Widget.objects.aggregate(total_weight=models.Sum('weight'))

print(f'count = {total}')
print(f'heavy widgets = {heavy}')
print(f'sum(weight) = {agg}')
assert total == 3
assert heavy == ['beta', 'gamma']
assert agg['total_weight'] == 60
